# Data Viualisation - Analysis Report

In this phase we will analyse in details the singular features of the datasets and their correlations, and we will try to answer the questions we formulated through the use of graphic representations. Looking at graphics provides prospective on our hypothesis, allowing us to confirm or discard them.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import functions as fun

In [ ]:
countries_df = pd.read_csv('../../Data/DataFinal/countries_v2.csv')
genres_df = pd.read_csv('../../Data/DataFinal/genres_v2.csv')
languages_df = pd.read_csv('../../Data/DataFinal/languages_v2.csv')
movies_df = pd.read_csv('../../Data/DataFinal/movies_v2.csv', parse_dates=['date'], low_memory=False)
oscar_df =  pd.read_csv('../../Data/DataFinal/the_oscar_awards_v2.csv', parse_dates=['year_ceremony'],)
reviews_df = pd.read_csv('../../Data/DataFinal/rotten_tomatoes_reviews_v2.csv', parse_dates=['review_date'])

In [ ]:
plt.style.use('ggplot')

## Which Countries Made the Most Acclaimed Films?

In this first section, we would like to conduct a deeper analysis on the Countries that made the films we're studying. In particular, we will analyse the number of Oscar nominations and wins per Country, as well as their films' review scores, to see which Nations were able to stand out the most and produce the most loved films.

In [ ]:
countries_movies_df = fun.countries_movies_merge(countries_df, movies_df)
countries_oscar_df = fun.countries_oscar_merge(countries_movies_df, oscar_df)
countries_oscar_year_df = fun.countries_oscar_year_merge(countries_movies_df, oscar_df)
countries_reviews_df = fun.countries_reviews_merge(countries_movies_df, reviews_df)

### Which Countries have the most Oscar nominations?

In [ ]:
# Count how many times each country appears in the merged DataFrame
country_counts = countries_oscar_df['country'].value_counts()

temp_df = country_counts.reset_index()
temp_df.columns = ['country', 'count']

fig = px.choropleth(
    temp_df,
    locations="country",
    locationmode="country names",
    color="count",
    color_continuous_scale="Viridis",
    range_color=[0, 5000],
    labels={'count': 'Oscar Nominations'}
)
fig.update_layout(margin={"r":10,"t":40,"l":10,"b":40})
fig.show()

Before commenting this map, let's create one showcasing the Countries with the most Oscar Wins and then let's compare them.

### Which Countries have the most Oscar Wins?

In [ ]:
# Filter only rows where 'winner' is True
winners_only_df = countries_oscar_df[countries_oscar_df['winner'] == True]
country_counts = winners_only_df['country'].value_counts()

temp_df = country_counts.reset_index()
temp_df.columns = ['country', 'count']

fig = px.choropleth(
    temp_df,
    locations="country",
    locationmode="country names",
    color="count",
    color_continuous_scale="Viridis",
    labels={'count': 'Oscar Wins'}
)
fig.update_layout(margin={"r":10,"t":40,"l":10,"b":40})
fig.show()

Now let's compare the two:

* In the first chart, we have a world map where, following the referenced scale to the right, the colour of each Country represents the number of films produced there that got nominated to the Oscars Awards.
The **USA**, in yellow, is the Country with the most nominated films, over 14 thousand, while the **United Kingdom** and **France** fall in the second and third positions respectively. While we lack data for some of them, most of the Countries have at least some nominations, even if their numbers are low, as showed by the hegemony of purple on the map.

* In the second chart, the map underlines how there is a significant gap between the first and the rest of the Countries. We can notice immediately that the **USA**, in yellow, is once again in top position, being the Country with the most Oscar-winning films. The only other Nation not coloured in purple is the **UK**, with about 900 wins, while the vast majority of the others have less than 300. Moreover, looking at the African continent, we notice that there are some Countries that, despite having some nominations, don't have any winnings.

The strong lead had by the USA shouldn't surprise us, as it is the Country that founded and hosts the Ceremony each year.

### Correlation between Oscar nominations and wins by Country:

In [ ]:
country_stats_df = fun.nom_win_number_c(countries_oscar_df)
country_stats_df = country_stats_df[country_stats_df['Wins'] > 0] # Include only countries with at least one win

# Interactive Bubble Plot
fig = px.scatter(country_stats_df,
                 x='Nominations',
                 y='Wins',
                 size='Nominations',
                 color='country',
                 hover_data={'Nominations': True, 'Wins': True},
                 title='Nominations vs Wins by Country')

fig.update_layout(
    margin={"r":10,"t":40,"l":10,"b":40},
    xaxis_title="Number of Nominations",
    yaxis_title="Number of Wins",
)
fig.show()

As we could imagine, there appears to be a clear correlation between the number of Oscar nominations and wins, with the **USA** easily being first in both categories, and the **UK** following. The relationship between nominations and wins appears to be roughly proportional for most countries, suggesting a consistent win rate.

Despite this, there still are some interesting observations shown by this plot. For example, **Italy** and **Japan** have a very similar number of nominations, but the latter has almost 20 wins more than the former.

The legend shows many Countries (Afghanistan, Albania, Algeria, etc.), though many of these smaller Countries' data points are difficult to distinguish in the crowded bottom-left section: this is why we made a dynamic chart - it's possible to uncheck, for example, the USA, to be able to zoom the view towards the other Countries.

### Which Country won the most Oscar Awards each year?

In [ ]:
countries_oscar_year_df['year_ceremony'] = pd.to_datetime(countries_oscar_year_df['year_ceremony'], errors='coerce').dt.year
winners_df = countries_oscar_year_df[countries_oscar_year_df['winner'] == True].copy()
# Get top 20 countries by total wins
top_countries = winners_df['country'].value_counts().nlargest(20).index
sorted_countries = sorted(top_countries)
filtered_df = winners_df[winners_df['country'].isin(top_countries)]
# Count wins per year per country
wins_per_year = filtered_df.groupby(['year_ceremony', 'country']).size().reset_index(name='wins')

# Plot Circle timeline
fig = px.scatter(
    wins_per_year,
    x='year_ceremony', y='country',
    size='wins', color='country',
    category_orders={'country': sorted_countries},
    title='Oscar Wins Per Year by Top 20 Countries', labels={'year_ceremony': 'Year', 'wins': 'Number of Wins'},
    width=1200, height=700
)
fig.update_traces(marker=dict(line=dict(width=1, color='DarkSlateGrey')))
fig.update_layout(xaxis=dict(dtick=1,tickfont=dict(size=8),range=[1927, 2025]))
fig.show()

 This chart provides a visualization of the distribution of Oscar wins across multiple Countries over time. The x-axis represents the years, while the y-axis lists the top 20 Countries by total wins, each distinguished by a unique color. The size of the bubbles indicates the number of wins, with larger bubbles representing more wins in that year.

 * The **United States** consistently show a high volume of wins, particularly in recent decades, as evidenced by the dense concentration of large bubbles. This doesn't surprise us, as this Country is the one with the most Wins overall, and the one hosting the Ceremony each year.
 * Countries like **France**, the **UK**, **Italy** and **Japan** also exhibit notable activity, but with less consistency and fewer overall wins compared to the USA.
 * Countries such as **Russia**, **China**, and **Brazil** display sporadic wins with smaller bubbles, indicating less frequent but still noteworthy achievements.
 * There are some Countries that show interesting behaviours: **Australia** and **Sweden**, for example, have some periods with a lot of wins and others completely dark, with little to no activity.

### Which Countries have the films with the highest review score?

In [ ]:
country_scores = countries_reviews_df.groupby('country', as_index=False)['normalized_score'].mean()

fig = px.choropleth(
    country_scores,
    locations="country",
    locationmode="country names",
    color="normalized_score",
    color_continuous_scale="Viridis",
    labels={'normalized_score': 'Avg Score (0-10)'}
)
fig.update_layout(margin={"r":10,"t":40,"l":10,"b":40})
fig.show()

At a first glance, this map could surprise us, as it shows that there is no direct correlation between the Oscar Awards selectio and Rotten Tomatoes ratings. The above chart is, indeed, very different from the previous two: it shows some African Countries, such as **Libya**, **Somalia** and **South Sudan**, as the ones with the most appreciated movies.

This data is interesting, as it clearly shows how the Countries with the largest film industries (such as the USA, India and China) do not necessarily have the highest average ratings. Smaller film industries in North Africa and the Middle East seem to get better reviews or audience response on average.

The average score for the **USA** is 6.23, which doesn't put it at the bottom of the scale, but is certainly far from the top. We can imagine that the USA can produce thousands of films each year, some of which deserve awards, but many of which are rather average.

**_FINAL THOUGHTS_**

Examining these charts reveals an interesting perspective on the global film industry.

The **United States** leads in Oscar nominations and wins, yet this does not necessarily reflect the quality of films produced. Smaller film industries in **African** Countries like **Libya** and **Somalia** achieve higher average ratings than many Hollywood films. Countries such as **Japan** demonstrate efficiency by winning more Oscars from fewer nominations compared to nations like **Italy**. Moreover, Countries like **Australia** and **Sweden** have seen periods of Oscar success followed by quieter times.

Overall, this analysis helped us to highlight how Oscar wins do not always align with the quality of films; the best movies, or at least the ones preferred by critic on Rotten Tomatoes, are not always the ones winning the most awards.

## Which Genres are Most Popular and Acclaimed?

In [ ]:
genres_movies_df = fun.genres_movies_merge(genres_df, movies_df)
genres_oscar_df = fun.genres_oscar_merge(genres_movies_df, oscar_df)
genres_countries_df = fun.genres_countries_merge(genres_movies_df, countries_df)
genres_reviews_df = fun.genres_reviews_merge(genres_movies_df, reviews_df)

### Which genres have the most Oscar nominations? And which have the most wins?

In [ ]:
top_genres = genres_oscar_df['genre'].value_counts()
#winners_only_df = genres_oscar_df[genres_oscar_df['winner'] == True]
top_genres_win = genres_oscar_df[genres_oscar_df['winner'] == True]['genre'].value_counts()

# Sort the values in descending order (largest to smallest)
top_genres = top_genres.sort_values(ascending=True)
top_genres_win = top_genres_win.sort_values(ascending=True)
viridis_colours = plt.cm.viridis(np.linspace(0, 1, 19))[::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
#Donut Chart 1
axes[0].pie(top_genres, labels=top_genres.index, colors=viridis_colours, startangle=-270, wedgeprops={'width': 0.5}, pctdistance=0.85,
        textprops={'fontsize': 7, 'fontweight': 'bold'})
axes[0].set_title("Top Genres by Number of Oscar Nominations", fontsize=12, fontweight='bold')
#Donut Chart2
axes[1].pie(top_genres_win, labels=top_genres_win.index, colors=viridis_colours, startangle=-270, wedgeprops={'width': 0.5}, pctdistance=0.85,
        textprops={'fontsize': 7, 'fontweight': 'bold'})
axes[1].set_title("Top Genres by Number of Oscar Wins", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

While these two plots look very similar at first glance, they actually show some meaningful differences.

**Drama** dominates both nominations and wins by a significant margin, represented by the large purple section in both charts.
The other top genres, **Romance** and **Comedy**, have a very similar percentage of nominations and wins. The proportions appear very similar between nominations and wins, suggesting that most genres convert nominations to wins at roughly similar rates.

But, if we look into the less rewarded genres we see that there are some differences, for example, **Animation** films have more wins than documentaries, despite having slightly fewer nominations.

However, in the end, the consistency between the two charts suggests that the Academy's genre preferences remain _fairly consistent from nomination to final selection of winners_.



### Correlation between Oscar nominations and wins by genre:

In [ ]:
genre_stats_df = fun.nom_win_number_g(genres_oscar_df)

# Interactive Bubble Plot
fig = px.scatter(genre_stats_df,
                 x='Nominations',
                 y='Wins',
                 size='Nominations',
                 color='genre',
                 hover_data={'Nominations': True, 'Wins': True},
                 title='Correlation between Nominations and Wins by Genre')

fig.update_layout(
    margin={"r":10,"t":40,"l":10,"b":40},
    xaxis_title='Number of Nominations',
    yaxis_title='Number of Wins'
)

As we saw from the previous chart, the **Drama** genre (pink bubble) has by far the most nominations and wins. It has around 18,000 nominations and 3,800 wins, which significantly outpaces other genres. This reflects how critically favoured drama is in awards.

Genres like **Comedy**, **Romance**, **Documentary** and **Animation** are in the middle of the cluster.

Some genres (like **TV Movie**, **Horror**, and **Fantasy**) may have a decent number of nominations but don’t win as often. This is interesting because it outlines the under-recognition of genres like Horror or Science Fiction despite commercial popularity.

### Which genres have the films with the highest review score?

In [ ]:
# We'll make 2 Violin Plots since there are so many genres
unique_genres = sorted(genres_reviews_df['genre'].dropna().unique())
first_half = unique_genres[:9]
second_half = unique_genres[9:]
plot_df1 = genres_reviews_df[genres_reviews_df['genre'].isin(first_half)]
plot_df2 = genres_reviews_df[genres_reviews_df['genre'].isin(second_half)]

fig, axes = plt.subplots(2, 1, figsize=(15,16))
# First plot (First 9 genres) on the first axis (axes[0])
sns.violinplot(
    x='genre',
    y='normalized_score',
    data=plot_df1,
    inner='quart',
    hue='genre',
    palette='cubehelix',
    ax=axes[0]
)
axes[0].set_title('Rotten Tomatoes Scores by Genre (1–9)', fontsize=16)
axes[0].set_xlabel('Genre', fontsize=14)
axes[0].set_ylabel('Normalized Score (out of 10)', fontsize=14)
axes[0].tick_params(axis='x', rotation=45)

# Second plot (Next 9 genres) on the second axis (axes[1])
sns.violinplot(
    x='genre',
    y='normalized_score',
    data=plot_df2,
    inner='quart',
    hue='genre',
    palette='cubehelix',
    ax=axes[1]
)
axes[1].set_title('Rotten Tomatoes Scores by Genre (10–18)', fontsize=16)
axes[1].set_xlabel('Genre', fontsize=14)
axes[1].set_ylabel('Normalized Score (out of 10)', fontsize=14)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

These violin plots give us a general idea, but let's create one more plot to see the differences in scores in detail.

In [ ]:
genre_avg_scores = genres_reviews_df.groupby('genre')['normalized_score'].mean().sort_values(ascending=False)

# Bar plot
plt.figure(figsize=(12, 8))
sns.barplot(
    x=genre_avg_scores.index,
    y=genre_avg_scores.values,
    hue = genre_avg_scores.index,
    palette='viridis'
)

plt.title('Average Rotten Tomatoes Score by Genre', fontsize=16)
plt.xlabel('Genre', fontsize=14)
plt.ylabel('Average Score (out of 10)', fontsize=14)
plt.xticks(rotation=45)
plt.ylim(0, 10)
plt.tight_layout()
plt.show()

As this barplot confirms, there isn't much of a difference between the average review scores of the different genres. We can see how **Historical** movies are the genre with the highest score, and critics appreciate **Action** films the least, but overall they all scored similarly on Rotten Tomatoes.
The lowest-rated genres still maintain average scores of around 6 out of 10, indicating that most films across all genres tend to receive moderately positive reviews on Rotten Tomatoes.

### Which genres are most popular in each Country?

Since the number of Countries is quite large, we will be analyzing only the top 20 by number of films.

In [ ]:
pivot_df = fun.pivot_count_gen(genres_countries_df, reviews_df)
# Heatmap
plt.figure(figsize=(14, 8))
sns.heatmap(pivot_df,
            cmap='viridis_r',
            annot=True,
            fmt=".2f",
            linewidths=0.5)
plt.title('Average Rotten Tomatoes Score by Country and Genre')
plt.xlabel('Genre')
plt.ylabel('Country')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In this chart, each cell represents the average Rotten Tomatoes score for films from a specific country (rows) in a specific genre (columns).
We notice immediately that:
* The **Philippines** has the highest single score (8.39) for **History** films.
* **Hong Kong's Music** films (7.99) and **Spain's Music** films (8.00) score exceptionally well.
* The genre **Western** has a really low score in **South Korea**, **USSR** and **Sweden**.
* Some cells appear blank (dark grey), indicating insufficient data for those country-genre combinations.

This visualization demonstrates that film quality varies significantly not just by genre, but by the intersection of country and genre, with each nation having distinct cinematic strengths and weaknesses.


### How has the genres popularity changed over the years?

In [ ]:
temp_gen_rev_df = genres_reviews_df
# Extract year from date
temp_gen_rev_df['year'] = temp_gen_rev_df['date'].dt.year

genres_grouped_df = (
    temp_gen_rev_df
    .groupby(['year', 'genre'])['normalized_score']
    .mean()
    .reset_index() )

fig = px.line(
    genres_grouped_df,
    x='year',
    y='normalized_score',
    color='genre',
    title="Average Rotten Tomatoes Score by Genre Over Time",
    labels={
        'normalized_score': 'Average Score',
        'year': 'Year',
        'genre': 'Genre' },
    markers=True
)

fig.update_traces(line=dict(width=1))  # thinner lines
fig.update_layout(
    height=500,
    title_font=dict(size=20),
    legend_title_text='Genre',
    yaxis=dict(range=[1, 10])
)
fig.show()

With this chart, we are calculating the average scores of each genre per year and looking at their trends over time.
We notice pretty interesting observations:
* there is a significant volatility in ratings across all genres throughout the entire period.
* scores generally range between 4 and 9 out of 10, with most clustering in the 5.5-7.5 range.
* there appears to be a period of higher scores and greater fluctuation between roughly 1920-1960.
* more extreme outliers (mostly very low scores) appear in the earlier decades than in more recent years.
* some isolated extremely low scores appear occasionally, particularly for Documentary (dipping to around 2.5 in the early 2000s) and Horror (dropping to around 3 near 2020).

_**FINAL THOUGHTS**_

Film analysis shows that while **Drama** dominates the Academy Awards (18,000 nominations, 3,800 wins), most genres receive similar Rotten Tomatoes ratings (6-7/10). **Historical** film ratings show significant volatility since 1900, with higher scores and greater fluctuations between 1920 and 1960. Country-genre combinations lead to unique patterns of excellence, such as the exceptional **Historical** films of the Philippines (8.39) and the poor reception of **Westerns** in South Korea and the USSR. Despite their commercial popularity, genres such as **Horror** and **Fantasy** have struggled to convert nominations into wins. This highlights the persistent discrepancy between critical recognition, audience preferences and the strengths of regional filmmakers in global cinema.


## What is the Recipe to Make a Great Film?

After taking a deeper look into which genres and which Countries have been most successful over time, we'd like to analyse what the most acclaimed films have in common and what are the key ingredients for a film to be remembered as great.

We will now concentrate on the study of movie lengths and languages, as well as the different perceptions Rotten Tomatoes critics and the Oscar Awards jury have of the same films, to discover which ones are more interesting to the public and, by the end of our analysis, to get a clear summary of all the best qualities to have, in order to obtain the recepie for a, in theoty, successful film.

### What's the correlation between Oscar Wins and reviews scores?

First of all, let's see if Rotten Tomatoes critics and the Oscar Awards jury have different perceptions of the same movies.

We will put the Top 10 films for number of Oscar Awards wins and the Top 10 films with the highest Rotten Tomatoes average reviews on 3 Heatmaps side by side, and we will compare the scores and the number of Oscars.

In [ ]:
map_data_df = fun.map_data_rev(oscar_df, reviews_df)

# Prepare heatmap data
z_nominations = map_data_df[['nominations']].values
z_wins = map_data_df[['wins']].values
z_scores = map_data_df[['normalized_score']].values
y_labels = map_data_df['film_label'].tolist()

# 3 Heatmaps
fig = go.Figure()
# Nominations heatmap
fig.add_trace(go.Heatmap( z=z_nominations, x=['Nominations'], y=y_labels, text=z_nominations.astype(str), texttemplate="%{text}", colorscale='Viridis', colorbar=dict(title='Oscar Nominations', len=0.3, y=0.85), hoverinfo='skip'))

# Wins heatmap
fig.add_trace(go.Heatmap( z=z_wins, x=['Oscar Wins'], y=y_labels, text=z_wins.astype(str), texttemplate="%{text}", colorscale='Viridis', colorbar=dict(title='Wins', len=0.3, y=0.5), xaxis='x2', yaxis='y2', hoverinfo='skip'))

# Review scores heatmap
fig.add_trace(go.Heatmap( z=z_scores, x=['Review Score'], y=y_labels, text=z_scores.round(2).astype(str), texttemplate="%{text}", colorscale='Viridis', colorbar=dict(title='Review Score', len=0.3, y=0.15), xaxis='x3', yaxis='y3', hoverinfo='skip'))

fig.update_layout( xaxis=dict(domain=[0, 0.25]), yaxis=dict(domain=[0, 1], showticklabels=True),
    xaxis2=dict(domain=[0.35, 0.6]), yaxis2=dict(domain=[0, 1], showticklabels=False),
    xaxis3=dict(domain=[0.7, 0.95]), yaxis3=dict(domain=[0, 1], showticklabels=False),
    height=800,
    width=1150,
    title='Heatmaps of Nominations, Wins, and Review Scores for Top Films',
    margin=dict(l=200)
)
fig.show()

We can immediately notice that, if a film was nominated and won multiple Oscar Awards, it usually has a great score on Rotten Tomatoes.

This has its sole exception in `Ben-Hur`, which, despite winning 11 Oscar prizes, scored just 5.38 on the critiques reviews. This might be due to several reasons. Firstly, we need to consider the plot of the film, which, in short, involves a Jewish prince betrayed by a Roman friend as he seeks revenge and redemption during the time of Christ. This film is mostly known for its themes of betrayal, faith, justice, and forgiveness.
Reasons why such a low score for an Oscar-winning film:
* **Religious Overtones**: The Christian themes are not liked the same way by the modern global audience.
* **Old-fashioned Film**: Today's viewers are used to CGI-heavy, fast-paced films, while `Ben-Hur` is slow and overlong, especially compared to today’s tighter editing styles.
* **Legacy vs. Entertainment**: Some viewers rate based on enjoyment alone, not legacy, artistry, or historical importance.

The contrary can't be said, we can see how most of the Top 10 Review-Scored films don't have any Oscar Awards nominations or wins, except just 4 of them. The Godfather distinguishes itself not only by scoring 9.71 on Rotten Tomatoes but also by being nominated in 11 categories and winning 3 of those.

### Correlation between duration and public interest:

Let's check whether the films with the longest or shortest duration captured the public interest the most.

In [ ]:
duration_grouped = fun.duration_merge(movies_df, reviews_df)

# Interactive Bubble Plot
fig = px.scatter(duration_grouped,x='avg_score',y='duration',
    size='review_count',
    hover_name='original_title',
    labels={'avg_score': 'Average Score', 'duration': 'Duration (minutes)', 'review_count': 'Number of Reviews'},
    color='review_count',
    color_continuous_scale='Viridis',
    title='Correlation between Rating Score, Number of Reviews and Duration',
    width=1000,
    height=700
)
fig.update_layout(xaxis_title="Rating Score", yaxis_title="Duration")
fig.show()

The analysis of the chart provides several important insights:
* **Duration Is Mostly Between 80–130 Minutes**: most films tend to fall within the 90 to 120-minute range, indicating that this duration is generally preferred by audiences. Very few films go beyond 200 minutes.
* **Outliers in Duration**: a limited number of films surpass 500 minutes, indicating extraordinarily long films. These films don’t appear to attract many reviews or ratings, based on the small dots.
* **Highly Reviewed Films Have Mid-to-High Ratings**: the largest and brightest bubbles are generally located around ratings of 7 to 8. This suggests that better-rated films attract more reviews.
* **Low-Rated Films (<4) Get Fewer Reviews**: there are fewer large or bright dots in the lower-left portion of the chart (low score, short/average duration), showing that poorly rated films receive less attention.

In conclusion, while there isn’t a strong correlation between film duration and ratings, the data suggests that moderately long films (90 to 120 minutes) paired with higher ratings (around 7 to 8) can optimize audience engagement.

### Which films aroused the most public interest?

In [ ]:
discussed = reviews_df.groupby('movie_title').size().reset_index(name='Review Count')
# Selecting the films with the highest number of reviews
most_discussed = discussed.sort_values('Review Count', ascending=False).head(15)
most_discussed['movie_title'] = most_discussed['movie_title'].str.title()

# Interactive Bar Plot
fig_most_discussed = px.bar(most_discussed, x='Review Count', y='movie_title', orientation='h',
                            title="Top 15 Films by Number of Reviews",
                            labels={'Review Count': 'Number of Reviews', 'movie_title': 'Film Title'},
                            color='Review Count',
                            color_continuous_scale='Plasma')

fig_most_discussed.update_layout(yaxis={'categoryorder':'total ascending'},
                                 xaxis_title="Number of Reviews",
                                 yaxis_title="Film Title",
                                 margin=dict(l=150))
fig_most_discussed.show()

The chart shows the "Top 15 Films by Number of Reviews" displayed as a horizontal bar chart.
The top 3 is composed of:
* **Halloween** (approximately 400 reviews) - shown with a yellow bar;
* **The Lion King** (approximately 390 reviews) - shown with an orange bar;
* **A Star Is Born** (approximately 380 reviews) - shown with a light pink/coral bar;

The remaining films on the list have between approximately 320-360 reviews, with a gradient of purple colours as the number decreases.
Many popular franchise films appear in this list, including multiple Marvel films, Star Wars, and Disney productions, suggesting these types of films tend to attract people's attention and get more reviews.

### What are the languages used the most in top rated films?

Finally, let's see what the most used languages are among the top 1.000 rated movies, so that we can get a general idea on which languages can help make a movie popular and acclaimed.

In [ ]:
lang_movie_df = fun.language_movies_merge(languages_df, movies_df)
lang_usage = fun.top15_language_trend(lang_movie_df)

fig = px.line(lang_usage,
              x='year',
              y='film_count',
              color='language',
              title="Top 15 Language Usage in Films Over Time",
              labels={'film_count': 'Number of Films', 'year': 'Year', 'language': 'Language'},
              markers=True)
fig.show()

As we can see, **English** is the most used language in films. This data makes sense since the USA produced the highest number of films and the most proficient actors in our dataset are Americans.

It's curious how, after the 2000s, we can see an increase in the usage of other languages too. Especially **French** and **Spanish**.

From Asia, we notice a language that is not really known, **Tagalog**. It's an Austronesian language spoken as a first language by the ethnic Tagalog people, who make up a quarter of the population of the Philippines.

Still from Asia, we notice **Japanese**, **Chinese** and **Korean**. As time passes, these countries start to produce well-known movies.

Relevant is also **Hindu**, knowing the fame of Bollywood films in South-East Asia.

CONCLUSIONI FINALI tirando le somme di tutta l'analisi complessiva